In [ ]:
import json
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except Exception:
    SMOTE = None
    ImbPipeline = None
METRIC_NAMES = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'AUROC', 'AUPRC']
TREE_MODELS = {'XGBoost', 'LightGBM', 'CatBoost', 'RandomForest'}

In [ ]:
class RecommendationConfig:
    def __init__(self, train_path='../../../datasets/train_set.csv', 
                 new_path='../../../datasets/external-1.csv', 
                 save_dir='recommendation', 
                 encoding_candidates=('utf-8', 'utf-8-sig', 'gbk', 'gb2312'), 
                 target='Survival', id_col='case_id', 
                 selected_groups=None, selected_cols=None, feature_selection_dir=None, selected_set_id=None, 
                 deploy_model_names=None, probability_aggregation='mean', cv=5, random_state=42, use_smote=False, 
                 smote_ratio=0.8, threshold_strategy='youden', candidate_library_mode='restricted', 
                 min_skeleton_count=5, max_drug_count=3, top_n_templates_per_skeleton=3, 
                 allow_zero_regimen=False, top_k=3, treatment_labels=None):
        self.train_path = train_path
        self.new_path = new_path
        self.save_dir = save_dir
        self.encoding_candidates = encoding_candidates
        self.target = target
        self.id_col = id_col
        self.selected_groups = selected_groups
        self.selected_cols = selected_cols
        self.feature_selection_dir = feature_selection_dir
        self.selected_set_id = selected_set_id
        self.deploy_model_names = deploy_model_names if deploy_model_names is not None else ['XGBoost']
        self.probability_aggregation = probability_aggregation
        self.cv = cv
        self.random_state = random_state
        self.use_smote = use_smote
        self.smote_ratio = smote_ratio
        self.threshold_strategy = threshold_strategy
        self.candidate_library_mode = candidate_library_mode
        self.min_skeleton_count = min_skeleton_count
        self.max_drug_count = max_drug_count
        self.top_n_templates_per_skeleton = top_n_templates_per_skeleton
        self.allow_zero_regimen = allow_zero_regimen
        self.top_k = top_k
        self.treatment_labels = treatment_labels if treatment_labels is not None else {'colistin_cms_daily_freq': 'CMS', 'polymyxin_b_daily_freq': 'PMB', 'colistin_sulfate_daily_freq': 'CS', 'carbapenem_daily_dose': 'Carbapenem', 'sulbactam_daily_dose': 'Sulbactam', 'tigecycline_daily_dose': 'Tigecycline', 'minocycline_daily_dose': 'Minocycline', 'vancomycin_daily_dose': 'Vancomycin', 'eravacycline_daily_dose': 'Eravacycline', 'aminoglycoside_daily_dose': 'Aminoglycoside'}

    def to_dict(self):
        return {'train_path': self.train_path, 
                'new_path': self.new_path, 
                'save_dir': self.save_dir, 
                'encoding_candidates': self.encoding_candidates, 
                'target': self.target, 
                'id_col': self.id_col, 
                'selected_groups': self.selected_groups, 
                'selected_cols': self.selected_cols, 
                'feature_selection_dir': self.feature_selection_dir, 
                'selected_set_id': self.selected_set_id, 
                'deploy_model_names': self.deploy_model_names, 
                'probability_aggregation': self.probability_aggregation, 
                'cv': self.cv, 'random_state': self.random_state, 
                'use_smote': self.use_smote, 
                'smote_ratio': self.smote_ratio, 
                'threshold_strategy': self.threshold_strategy, 
                'candidate_library_mode': self.candidate_library_mode, 
                'min_skeleton_count': self.min_skeleton_count, 
                'max_drug_count': self.max_drug_count, 
                'top_n_templates_per_skeleton': self.top_n_templates_per_skeleton, 
                'allow_zero_regimen': self.allow_zero_regimen, 
                'top_k': self.top_k, 
                'treatment_labels': self.treatment_labels}


In [ ]:
# =========================================================
# Utilities reused from your notebook style
# =========================================================
def read_csv_flexible(path, encoding_candidates):
    last_error = None
    for enc in encoding_candidates:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_error = e
            continue
    raise RuntimeError(f'Failed to read CSV: {path}\nLast error: {last_error}')


# -----------------------------
# feature selection
# -----------------------------
def feature_selection(df, target):
    polyType_cols = ['colistin_cms_daily_freq', 'polymyxin_b_daily_freq', 'colistin_sulfate_daily_freq']
    Combination_medication_cols = ['carbapenem_daily_dose', 'sulbactam_daily_dose', 'tigecycline_daily_dose', 'minocycline_daily_dose', 'vancomycin_daily_dose', 'eravacycline_daily_dose', 'aminoglycoside_daily_dose']
    medication_features = polyType_cols + Combination_medication_cols
    time_features = ['Pre_Hospital_Days', 'Pre_ICU_Days']
    base_cols = ['Age', 'Gender', 'BMI']
    comorb_cols = ['Diabetes Mellitus', 'Hypertension', 'Heart Disease', 'Stroke', 'Malignant Tumor', 'Chronic Kidney Disease', 'Chronic Liver Disease', 'COPD', 'Comorb_other']
    df = df.copy()
    existing_comorb = [c for c in comorb_cols if c in df.columns]
    if existing_comorb:
        df[existing_comorb] = df[existing_comorb].fillna(0)
        df['Comorb_count'] = df[existing_comorb].sum(axis=1)
    else:
        df['Comorb_count'] = 0
    comorb_cols = comorb_cols + ['Comorb_count']
    immuno_cols = ['Use immunosuppressive agents', 'Neutrophil Reduction', 'HIV/AIDS', 'Post-Transplant Status', 'Chemotherapy/Radiation', 'immuno_Other']
    support_cols = ['Resp_support', 'Oxygen_concentration']
    pre_lab_cols = ['WBC', 'N_percent', 'L_count', 'PLT', 'CRP1', 'PCT1', 'D-d', 'Cr_baseline', 'eGFR1', 'RRT', 'ALT', 'AST', 'TB', 'ALB']
    dynamic_lab_cols = []
    lab_cols = pre_lab_cols + dynamic_lab_cols
    infection_cols = ['Infection_HAP', 'Infection_VAP']
    Coinfection_cols = ['Coinfection_G_Pos', 'Coinfection_G_Neg', 'Coinfection_Fungi']
    existing_coinf = [c for c in Coinfection_cols if c in df.columns]
    if existing_coinf:
        df[existing_coinf] = df[existing_coinf].fillna(0).astype(int)
    infection_cols = infection_cols + Coinfection_cols
    resistance_features = ['resistance_SXT', 'resistance_KAN', 'resistance_MIN', 'resistance_TGC', 'resistance_CFP-SUL', 'resistance_TOB']
    mapping = {'R': 1, 'I': 1, 'S': 0}
    for c in resistance_features:
        if c in df.columns:
            s = df[c].astype(str).str.strip().str.upper()
            df[c] = pd.to_numeric(s.map(mapping), errors='coerce')
    group_defs = {'Comorbidity': [c for c in comorb_cols if c in df.columns], 'Immunosuppression': [c for c in immuno_cols if c in df.columns]}
    if target == 'Target_Polymyxin':
        feature_cols = base_cols + time_features + comorb_cols + immuno_cols + support_cols + pre_lab_cols
        include_med = False
    else:
        feature_cols = base_cols + comorb_cols + immuno_cols + support_cols + lab_cols + infection_cols + resistance_features + polyType_cols + Combination_medication_cols
        include_med = True
    feature_cols = [c for c in feature_cols if c in df.columns]
    medication_features = [c for c in medication_features if c in feature_cols or c in df.columns]
    return (feature_cols, df, medication_features, group_defs, include_med)


In [ ]:
# -----------------------------
# group helpers (copied from your notebook logic)
# -----------------------------
def build_group_to_cols(feature_cols, group_defs):
    group_to_cols = {g: cols[:] for g, cols in group_defs.items()}
    grouped_cols = set(sum(group_defs.values(), []))
    for c in feature_cols:
        if c in grouped_cols:
            continue
        group_to_cols.setdefault(c, [c])
    return group_to_cols

def groups_to_feature_cols(groups, group_to_cols, medication_features=None, include_med=False):
    cols = []
    for g in groups:
        cols.extend(group_to_cols.get(g, [g]))
    cols = list(dict.fromkeys(cols))
    if include_med and medication_features:
        cols = list(dict.fromkeys(list(medication_features) + cols))
    return cols


# -----------------------------
# model utilities
# -----------------------------
def get_models(spw):
    return {'XGBoost': XGBClassifier(scale_pos_weight=spw, use_label_encoder=False, objective='binary:logistic', eval_metric='logloss', verbosity=0, random_state=42, n_jobs=1), 'LightGBM': LGBMClassifier(class_weight='balanced', verbosity=-1, random_state=42, n_jobs=1), 'CatBoost': CatBoostClassifier(auto_class_weights='Balanced', verbose=0, loss_function='Logloss', allow_writing_files=False, random_state=42, thread_count=1), 'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=1), 'LogisticRegression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000), 'SVM': SVC(class_weight='balanced', random_state=42, probability=True), 'KNN': KNeighborsClassifier(weights='distance', n_jobs=1)}

def is_tree(model_name):
    return model_name in TREE_MODELS

def create_pipeline(model, model_name, use_smote=False, smote_ratio=0.8):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if not is_tree(model_name):
        steps.append(('scaler', StandardScaler()))
    if use_smote:
        if ImbPipeline is None or SMOTE is None:
            raise ImportError('imblearn is required when use_smote=True')
        steps.append(('smote', SMOTE(random_state=42, sampling_strategy=smote_ratio)))
    steps.append(('clf', clone(model)))
    if use_smote:
        return ImbPipeline(steps)
    return Pipeline(steps)

In [ ]:
# =========================================================
# Feature-set resolution
# =========================================================
def load_selected_groups_from_search_outputs(feature_selection_dir, selected_set_id):
    details_path = os.path.join(feature_selection_dir, 'candidate_details-nogridsearch.csv')
    if not os.path.exists(details_path):
        raise FileNotFoundError(f'Cannot find: {details_path}')
    details_df = pd.read_csv(details_path, encoding='utf-8-sig')
    sub = details_df.loc[details_df['set_id'] == selected_set_id]
    if sub.empty:
        raise ValueError(f'selected_set_id={selected_set_id} not found in {details_path}')
    groups_str = sub.iloc[0]['groups']
    if not isinstance(groups_str, str) or not groups_str.strip():
        raise ValueError(f'Invalid groups for set_id={selected_set_id}')
    return groups_str.split('|')


# =========================================================
# Data alignment
# =========================================================
def add_missing_columns(df, columns, fill_value=np.nan):
    df = df.copy()
    for c in columns:
        if c not in df.columns:
            df[c] = fill_value
    return df

In [ ]:
# =========================================================
# OOF probability + threshold selection
# =========================================================
def find_best_threshold(y_true, y_proba, strategy='youden'):
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba, dtype=float)
    if strategy == 'f1':
        precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
        if len(thresholds) == 0:
            return 0.5
        f1 = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
        return float(thresholds[int(np.nanargmax(f1))])
    grid = np.unique(np.round(y_proba, 6))
    if grid.size == 0:
        return 0.5
    best_thr, best_score = (0.5, -np.inf)
    for thr in grid:
        pred = (y_proba >= thr).astype(int)
        sens = recall_score(y_true, pred, zero_division=0)
        spec = recall_score(1 - y_true, 1 - pred, zero_division=0)
        score = sens + spec - 1.0
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr


def get_oof_ensemble_probabilities(X, y, model_dict, deploy_model_names, cv, random_state, use_smote, smote_ratio, aggregation='mean'):
    if aggregation != 'mean':
        raise NotImplementedError('Only mean aggregation is currently implemented.')
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    oof = np.zeros(len(X), dtype=float)
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = (X.iloc[tr_idx], X.iloc[va_idx])
        y_tr = y.iloc[tr_idx]
        fold_probs = []
        for model_name in deploy_model_names:
            model = model_dict[model_name]
            pipe = create_pipeline(model, model_name, use_smote=use_smote, smote_ratio=smote_ratio)
            pipe.fit(X_tr, y_tr)
            fold_probs.append(pipe.predict_proba(X_va)[:, 1])
        oof[va_idx] = np.mean(np.vstack(fold_probs), axis=0)
    return oof

In [ ]:
# =========================================================
# Final model training
# =========================================================
def fit_deployment_models(X_train, y_train, all_models, deploy_model_names, use_smote, smote_ratio):
    fitted = {}
    for model_name in deploy_model_names:
        pipe = create_pipeline(all_models[model_name], model_name, use_smote=use_smote, smote_ratio=smote_ratio)
        pipe.fit(X_train, y_train)
        fitted[model_name] = pipe
    return fitted

def predict_positive_probability(fitted_models, X, aggregation='mean'):
    probs = []
    for _, pipe in fitted_models.items():
        probs.append(pipe.predict_proba(X)[:, 1])
    probs_arr = np.vstack(probs)
    if aggregation == 'mean':
        return probs_arr.mean(axis=0)
    raise NotImplementedError(f'Unsupported aggregation: {aggregation}')

In [ ]:
# =========================================================
# Candidate regimen library
# =========================================================
def _coerce_treatment_numeric(df, treatment_cols):
    df = df.copy()
    for c in treatment_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    return df

def _active_treatment_cols(row, treatment_cols):
    active = []
    for c in treatment_cols:
        v = row.get(c, 0)
        try:
            v = float(v)
        except Exception:
            v = 0.0
        if v > 0:
            active.append(c)
    return active

def _make_skeleton_name(active_cols, label_map):
    if not active_cols:
        return 'NO_DRUG'
    return '+'.join([label_map.get(c, c) for c in active_cols])

def _template_key_from_row(row, treatment_cols):
    return tuple((float(pd.to_numeric(row.get(c, 0), errors='coerce') if hasattr(pd, 'to_numeric') else row.get(c, 0)) for c in treatment_cols))

def build_candidate_regimen_library(train_df, treatment_cols, label_map, min_skeleton_count=5, top_n_templates_per_skeleton=3, max_drug_count=3, allow_zero_regimen=False, candidate_library_mode='restricted'):
    work = _coerce_treatment_numeric(train_df, treatment_cols).copy()
    skeleton_records = []
    for idx, row in work.iterrows():
        active_cols = _active_treatment_cols(row, treatment_cols)
        skeleton_name = _make_skeleton_name(active_cols, label_map)
        drug_count = len(active_cols)
        skeleton_records.append({'row_idx': idx, 'skeleton_name': skeleton_name, 'active_cols': tuple(active_cols), 'drug_count': drug_count, 'template_key': tuple((float(row.get(c, 0)) for c in treatment_cols))})
    skeleton_df = pd.DataFrame(skeleton_records)
    skeleton_summary_all = skeleton_df.groupby(['skeleton_name', 'drug_count', 'active_cols'], dropna=False).size().reset_index(name='train_count').sort_values(['train_count', 'skeleton_name'], ascending=[False, True]).reset_index(drop=True)
    skeleton_summary = skeleton_summary_all.copy()
    if not allow_zero_regimen:
        skeleton_summary = skeleton_summary.loc[skeleton_summary['drug_count'] > 0].copy()
    if candidate_library_mode not in {'restricted', 'full'}:
        raise ValueError(f'Unsupported candidate_library_mode: {candidate_library_mode}')
    if candidate_library_mode == 'restricted':
        skeleton_summary = skeleton_summary.loc[skeleton_summary['drug_count'] <= max_drug_count].copy()
        skeleton_summary = skeleton_summary.loc[skeleton_summary['train_count'] >= min_skeleton_count].copy()
    template_rows = []
    candidate_rows = []
    for _, sk_row in skeleton_summary.iterrows():
        sk_name = sk_row['skeleton_name']
        sub_idx = skeleton_df.loc[skeleton_df['skeleton_name'] == sk_name, 'row_idx']
        sub = work.loc[sub_idx, list(treatment_cols)].copy()
        if sub.empty:
            continue
        template_counts = sub.groupby(list(treatment_cols), dropna=False).size().reset_index(name='template_count').sort_values('template_count', ascending=False).reset_index(drop=True)
        if candidate_library_mode == 'restricted':
            template_counts = template_counts.head(top_n_templates_per_skeleton).reset_index(drop=True)
        for j, (_, tpl) in enumerate(template_counts.iterrows(), start=1):
            template_id = f'{sk_name}__tpl_{j:02d}'
            template_payload = {c: float(tpl[c]) for c in treatment_cols}
            template_rows.append({'skeleton_name': sk_name, 'template_id': template_id, 'drug_count': int(sk_row['drug_count']), 'train_skeleton_count': int(sk_row['train_count']), 'template_count': int(tpl['template_count']), **template_payload})
            nonzero_parts = []
            for c in treatment_cols:
                v = float(tpl[c])
                if v > 0:
                    nonzero_parts.append(f'{label_map.get(c, c)}={v:g}')
            regimen_name = ' | '.join(nonzero_parts) if nonzero_parts else 'NO_DRUG'
            candidate_rows.append({'candidate_id': f'cand_reg_{len(candidate_rows) + 1:03d}', 'skeleton_name': sk_name, 'template_id': template_id, 'drug_count': int(sk_row['drug_count']), 'regimen_name': regimen_name, 'train_skeleton_count': int(sk_row['train_count']), 'template_count': int(tpl['template_count']), **template_payload})
    candidate_df = pd.DataFrame(candidate_rows)
    template_df = pd.DataFrame(template_rows)
    skeleton_summary = skeleton_summary.reset_index(drop=True)
    return (candidate_df, skeleton_summary, template_df)

In [ ]:
# =========================================================
# Counterfactual feature construction
# =========================================================
def apply_regimen_to_patient(patient_row, regimen_row, treatment_cols):
    x = patient_row.copy()
    for c in treatment_cols:
        x[c] = float(regimen_row[c])
    return x

In [ ]:
# =========================================================
# Reporting helpers
# =========================================================
def make_observed_regimen_name(patient_row, treatment_cols, label_map):
    parts = []
    for c in treatment_cols:
        v = pd.to_numeric(patient_row.get(c, 0), errors='coerce')
        v = 0 if pd.isna(v) else float(v)
        if v > 0:
            parts.append(f'{label_map.get(c, c)}={v:g}')
    return ' | '.join(parts) if parts else 'NO_DRUG'

def summarize_patient_results(patient_df):
    summary = {'n_patients': int(len(patient_df)), 'mean_delta_probability_top1': float(patient_df['delta_probability_top1'].mean()) if len(patient_df) else np.nan, 'median_delta_probability_top1': float(patient_df['delta_probability_top1'].median()) if len(patient_df) else np.nan, 'flip_to_positive_rate': float(patient_df['flip_to_positive'].mean()) if len(patient_df) else np.nan}
    has_true = patient_df['true_label'].notna() if 'true_label' in patient_df.columns else pd.Series(dtype=bool)
    if len(patient_df) and has_true.any():
        neg_sub = patient_df.loc[patient_df['true_label'] == 0]
        summary['n_true_negative_outcomes'] = int(len(neg_sub))
        summary['simulated_rescue_rate_among_true_negatives'] = float(neg_sub['rec_top1_pred_label'].mean()) if len(neg_sub) else np.nan
    return summary

def compute_train_oof_metrics(y_true, y_proba, threshold):
    y_true_arr = np.asarray(y_true).astype(int)
    pred = (y_proba >= threshold).astype(int)
    return {'Accuracy': float(accuracy_score(y_true_arr, pred)), 'Precision': float(precision_score(y_true_arr, pred, zero_division=0)), 'Recall': float(recall_score(y_true_arr, pred, zero_division=0)), 'F1-score': float(f1_score(y_true_arr, pred, zero_division=0)), 'AUROC': float(roc_auc_score(y_true_arr, y_proba)), 'AUPRC': float(average_precision_score(y_true_arr, y_proba)), 'threshold': float(threshold)}


In [ ]:
# =========================================================
# Recommendation on new cohort
# =========================================================
def recommend_for_new_cohort(new_df, selected_cols, treatment_cols, fitted_models, threshold, config):
    candidate_df = config._candidate_df.copy()
    patient_rows = []
    long_rows = []
    for idx, row in new_df.iterrows():
        patient_id = row[config.id_col] if config.id_col in row.index else f'row_{idx}'
        observed_x = row[selected_cols].to_frame().T
        p_obs = float(predict_positive_probability(fitted_models, observed_x, config.probability_aggregation)[0])
        pred_obs = int(p_obs >= threshold)
        candidate_features = []
        for _, regimen_row in candidate_df.iterrows():
            cf_row = apply_regimen_to_patient(row[selected_cols], regimen_row, treatment_cols)
            candidate_features.append(cf_row)
        candidate_features_df = pd.DataFrame(candidate_features)[list(selected_cols)]
        p_all = predict_positive_probability(fitted_models, candidate_features_df, config.probability_aggregation)
        scores_df = candidate_df[['candidate_id', 'skeleton_name', 'template_id', 'regimen_name', 'drug_count', 'train_skeleton_count', 'template_count']].copy()
        scores_df['pred_probability'] = p_all
        scores_df['rank'] = scores_df['pred_probability'].rank(ascending=False, method='first').astype(int)
        scores_df = scores_df.sort_values(['pred_probability', 'rank'], ascending=[False, True]).reset_index(drop=True)
        scores_df.insert(0, 'patient_id', patient_id)
        topk = scores_df.head(config.top_k).reset_index(drop=True)
        top1 = topk.iloc[0]
        p_rec = float(top1['pred_probability'])
        pred_rec = int(p_rec >= threshold)
        true_label = row[config.target] if config.target in row.index and pd.notna(row[config.target]) else np.nan
        try:
            true_label = int(true_label) if pd.notna(true_label) else np.nan
        except Exception:
            true_label = np.nan
        obs_regimen_name = make_observed_regimen_name(row, treatment_cols, config.treatment_labels)
        delta_p = p_rec - p_obs
        patient_result = {'patient_id': patient_id, 'true_label': true_label, 'obs_regimen_name': obs_regimen_name, 'obs_pred_probability': p_obs, 'obs_pred_label': pred_obs, 'rec_top1_candidate_id': top1['candidate_id'], 'rec_top1_regimen_name': top1['regimen_name'], 'rec_top1_pred_probability': p_rec, 'rec_top1_pred_label': pred_rec, 'delta_probability_top1': delta_p, 'flip_to_positive': int(pred_obs == 0 and pred_rec == 1)}
        for j in range(config.top_k):
            if j < len(topk):
                rec = topk.iloc[j]
                patient_result[f'rec_top{j + 1}_regimen_name'] = rec['regimen_name']
                patient_result[f'rec_top{j + 1}_pred_probability'] = float(rec['pred_probability'])
            else:
                patient_result[f'rec_top{j + 1}_regimen_name'] = np.nan
                patient_result[f'rec_top{j + 1}_pred_probability'] = np.nan
        patient_rows.append(patient_result)
        long_rows.append(scores_df)
    patient_df = pd.DataFrame(patient_rows)
    long_df = pd.concat(long_rows, ignore_index=True) if long_rows else pd.DataFrame()
    return (patient_df, long_df)



In [ ]:
# =========================================================
# Main
# =========================================================
def run_recommendation_experiment(config):
    os.makedirs(config.save_dir, exist_ok=True)
    train_df_raw = read_csv_flexible(config.train_path, config.encoding_candidates)
    new_df_raw = read_csv_flexible(config.new_path, config.encoding_candidates)
    if config.id_col not in train_df_raw.columns:
        train_df_raw[config.id_col] = np.arange(len(train_df_raw))
    if config.id_col not in new_df_raw.columns:
        new_df_raw[config.id_col] = np.arange(len(new_df_raw))
    if config.target not in train_df_raw.columns:
        raise KeyError(f"Target column '{config.target}' not found in training data.")
    train_df_raw = train_df_raw.loc[train_df_raw[config.target].notna()].copy()
    train_df_raw[config.target] = train_df_raw[config.target].astype(int)
    feature_cols, train_df, medication_features, group_defs, include_med = feature_selection(
        train_df_raw,
        config.target,
    )
    group_to_cols = build_group_to_cols(feature_cols, group_defs)
    if config.selected_groups is None and config.selected_cols is None:
        if config.feature_selection_dir and config.selected_set_id:
            config.selected_groups = load_selected_groups_from_search_outputs(
                config.feature_selection_dir,
                config.selected_set_id,
            )
        else:
            raise ValueError(
                'Please provide either selected_groups / selected_cols, '
                'or set feature_selection_dir + selected_set_id.'
            )
    if config.selected_cols is not None:
        selected_cols = list(dict.fromkeys(config.selected_cols))
    else:
        selected_cols = groups_to_feature_cols(
            config.selected_groups,
            group_to_cols,
            medication_features=medication_features,
            include_med=include_med,
        )
    new_df = new_df_raw.copy()
    new_df = add_missing_columns(new_df, feature_cols, fill_value=np.nan)
    new_df = add_missing_columns(new_df, medication_features, fill_value=0)
    train_df = add_missing_columns(train_df, selected_cols, fill_value=np.nan)
    new_df = add_missing_columns(new_df, selected_cols, fill_value=np.nan)
    train_df = add_missing_columns(train_df, medication_features, fill_value=0)
    new_df = add_missing_columns(new_df, medication_features, fill_value=0)
    X_train = train_df[selected_cols].copy()
    y_train = train_df[config.target].astype(int).copy()
    scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    all_models = get_models(scale_pos_weight)
    missing_models = [m for m in config.deploy_model_names if m not in all_models]
    if missing_models:
        raise ValueError(f'Unsupported model(s): {missing_models}')
    oof_proba = get_oof_ensemble_probabilities(
        X=X_train,
        y=y_train,
        model_dict=all_models,
        deploy_model_names=config.deploy_model_names,
        cv=config.cv,
        random_state=config.random_state,
        use_smote=config.use_smote,
        smote_ratio=config.smote_ratio,
        aggregation=config.probability_aggregation,
    )
    threshold = 0.5
    train_oof_metrics = compute_train_oof_metrics(y_train, oof_proba, threshold)
    fitted_models = fit_deployment_models(
        X_train=X_train,
        y_train=y_train,
        all_models=all_models,
        deploy_model_names=config.deploy_model_names,
        use_smote=config.use_smote,
        smote_ratio=config.smote_ratio,
    )
    candidate_df, skeleton_summary_df, template_df = build_candidate_regimen_library(
        train_df=train_df,
        treatment_cols=medication_features,
        label_map=config.treatment_labels,
        min_skeleton_count=config.min_skeleton_count,
        top_n_templates_per_skeleton=config.top_n_templates_per_skeleton,
        max_drug_count=config.max_drug_count,
        allow_zero_regimen=config.allow_zero_regimen,
        candidate_library_mode=config.candidate_library_mode,
    )
    if candidate_df.empty:
        raise ValueError(
            'Candidate regimen library is empty. Relax min_skeleton_count or template filters.'
        )
    config._candidate_df = candidate_df
    patient_results_df, candidate_scores_long_df = recommend_for_new_cohort(
        new_df=new_df,
        selected_cols=selected_cols,
        treatment_cols=medication_features,
        fitted_models=fitted_models,
        threshold=threshold,
        config=config,
    )
    summary = summarize_patient_results(patient_results_df)
    candidate_df.to_csv(
        os.path.join(config.save_dir, 'candidate_regimen_library.csv'),
        index=False,
        encoding='utf-8-sig',
    )
    skeleton_summary_df.to_csv(
        os.path.join(config.save_dir, 'candidate_skeleton_summary.csv'),
        index=False,
        encoding='utf-8-sig',
    )
    template_df.to_csv(
        os.path.join(config.save_dir, 'candidate_template_summary.csv'),
        index=False,
        encoding='utf-8-sig',
    )
    patient_results_df.to_csv(
        os.path.join(config.save_dir, 'patient_level_recommendations.csv'),
        index=False,
        encoding='utf-8-sig',
    )
    candidate_scores_long_df.to_csv(
        os.path.join(config.save_dir, 'candidate_scores_long.csv'),
        index=False,
        encoding='utf-8-sig',
    )
    run_meta = {
        'config': {k: v for k, v in config.to_dict().items() if not k.startswith('_')},
        'selected_cols': selected_cols,
        'selected_groups': config.selected_groups,
        'medication_features': medication_features,
        'threshold': threshold,
        'train_oof_metrics': train_oof_metrics,
        'summary': summary,
    }
    with open(os.path.join(config.save_dir, 'run_meta.json'), 'w', encoding='utf-8') as f:
        json.dump(run_meta, f, ensure_ascii=False, indent=2)
    return {
        'run_meta': run_meta,
        'patient_results': patient_results_df,
        'candidate_scores_long': candidate_scores_long_df,
        'candidate_regimen_library': candidate_df,
        'candidate_skeleton_summary': skeleton_summary_df,
        'candidate_template_summary': template_df,
    }



In [ ]:
TRAINSET_PATH = '../../../datasets/train_set.csv'
EXTERNAL_PATH = '../../../datasets/external-1.csv'
SAVE_DIR = 'recommendation'
TARGET = 'Survival'
os.makedirs(SAVE_DIR, exist_ok=True)
config = RecommendationConfig(train_path=TRAINSET_PATH, new_path=EXTERNAL_PATH, save_dir=SAVE_DIR, 
                        target=TARGET, id_col='case_id', 
                        feature_selection_dir='../../../results/survival0203/feature_selection', selected_set_id='cand_16', 
                        deploy_model_names=['XGBoost'], cv=5, threshold_strategy='youden', candidate_library_mode='full', 
                        min_skeleton_count=3, max_drug_count=3, top_n_templates_per_skeleton=3, allow_zero_regimen=False, top_k=3)
results = run_recommendation_experiment(config)
